# Assignment 18 — Text Vectorization Techniques

**Dataset:** SMS Spam Collection Dataset  
**Kaggle Link:** https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset  
**Student:** Abhishek Thakare

## Imports

In [1]:
import os
import re
import nltk
import numpy as np
import pandas as pd

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Download required NLTK data
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Make sure the outputs folder exists
os.makedirs("outputs", exist_ok=True)

pd.set_option("display.max_columns", None)
print("All imports successful.")

All imports successful.


## Load & Clean the Dataset

In [2]:
# Load the raw CSV — it has extra unnamed columns we can drop
df = pd.read_csv("spam.csv", encoding="latin-1")

# Keep only the two meaningful columns and rename them
df = df[["v1", "v2"]].copy()
df.columns = ["label", "message"]

print(f"Dataset shape: {df.shape}")
print(f"\nLabel distribution:\n{df['label'].value_counts()}")
print(f"\nMissing values:\n{df.isnull().sum()}")
df.head()

Dataset shape: (5572, 2)

Label distribution:
label
ham     4825
spam     747
Name: count, dtype: int64

Missing values:
label      0
message    0
dtype: int64


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


## Text Preprocessing

Before vectorizing we clean each message:
- Lowercase
- Remove punctuation and numbers
- Tokenize
- Remove English stop words
- Lemmatize (reduce words to root form)

In [3]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()


def preprocess_text(text):
    """Clean, tokenize, remove stopwords, and lemmatize a single text string."""
    # 1. Lowercase
    text = str(text).lower()

    # 2. Remove everything except letters and spaces
    text = re.sub(r"[^a-zA-Z\s]", "", text)

    # 3. Tokenize into individual words
    tokens = word_tokenize(text)

    # 4. Drop stopwords and lemmatize what remains
    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in stop_words
    ]

    return " ".join(tokens)


df["clean_text"] = df["message"].apply(preprocess_text)

print("Sample before vs after cleaning:")
df[["message", "clean_text"]].head()

Sample before vs after cleaning:


,message,clean_text
0,"Go until jurong point, crazy.. Available only ...",go jurong point crazy available bugis n great ...
1,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entry wkly comp win fa cup final tkts st ...
3,U dun say so early hor... U c already then say...,u dun say early hor u c already say
4,"Nah I don't think he goes to usf, he lives aro...",nah dont think go usf life around though


---
# PART 1 — One-Hot Encoding (Text Level)

## Task 1: Manual One-Hot Encoding

We pick 5 sample sentences and manually build a one-hot matrix using plain Python dictionaries.

In [4]:
# Select 5 representative cleaned sentences
sample_sentences = df["clean_text"].head(5).tolist()

for i, s in enumerate(sample_sentences):
    print(f"Sentence {i}: {s}")

Sentence 0: go jurong point crazy available bugis n great world la e buffet cine got amore wat
Sentence 1: ok lar joking wif u oni
Sentence 2: free entry wkly comp win fa cup final tkts st may text fa receive entry questionstd txt ratetcs apply over
Sentence 3: u dun say early hor u c already say
Sentence 4: nah dont think go usf life around though


In [5]:
# Build the vocabulary — all unique words across the 5 sentences, sorted
all_words = " ".join(sample_sentences).split()
vocabulary = sorted(list(set(all_words)))

print(f"Vocabulary size: {len(vocabulary)}")
print(f"Vocabulary: {vocabulary}")

Vocabulary size: 53
Vocabulary: ['already', 'amore', 'apply', 'around', 'available', 'buffet', 'bugis', 'c', 'cine', 'comp', 'crazy', 'cup', 'dont', 'dun', 'e', 'early', 'entry', 'fa', 'final', 'free', 'go', 'got', 'great', 'hor', 'joking', 'jurong', 'la', 'lar', 'life', 'may', 'n', 'nah', 'ok', 'oni', 'over', 'point', 'questionstd', 'ratetcs', 'receive', 'say', 'st', 'text', 'think', 'though', 'tkts', 'txt', 'u', 'usf', 'wat', 'wif', 'win', 'wkly', 'world']


In [6]:
# Build one-hot vectors — 1 if word appears in sentence, 0 otherwise
one_hot_vectors = []

for sentence in sample_sentences:
    words_in_sentence = set(sentence.split())  # use set for fast lookup
    vector = [1 if word in words_in_sentence else 0 for word in vocabulary]
    one_hot_vectors.append(vector)

# Display as a DataFrame
manual_df = pd.DataFrame(one_hot_vectors, columns=vocabulary)

print(f"One-hot matrix shape: {manual_df.shape}  (sentences × unique words)")
manual_df

One-hot matrix shape: (5, 53)  (sentences × unique words)


,already,amore,apply,around,available,buffet,bugis,c,cine,comp,crazy,cup,dont,dun,e,early,entry,fa,final,free,go,got,great,hor,joking,jurong,la,lar,life,may,n,nah,ok,oni,over,point,questionstd,ratetcs,receive,say,st,text,think,though,tkts,txt,u,usf,wat,wif,win,wkly,world
0,0,1,0,0,1,1,1,0,1,0,1,0,0,0,1,0,0,0,0,0,1,1,1,0,0,1,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0
2,0,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,1,1,1,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,1,1,1,0,1,1,0,0,1,1,0,0,0,0,1,1,0
3,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0
4,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,1,0,0,0,0,0


In [7]:
# Save output
manual_df.to_csv("outputs/manual_one_hot.csv", index=False)
print("Saved: outputs/manual_one_hot.csv")

Saved: outputs/manual_one_hot.csv


## Task 2: One-Hot Encoding Using Scikit-learn

Scikit-learn's `MultiLabelBinarizer` can produce the same result from tokenized input.

In [8]:
# Tokenize each sentence into a list of words
tokenized_sentences = [sentence.split() for sentence in sample_sentences]

# Fit and transform
mlb = MultiLabelBinarizer()
encoded = mlb.fit_transform(tokenized_sentences)

# Wrap in a DataFrame with meaningful column names
sklearn_ohe = pd.DataFrame(encoded, columns=mlb.classes_)

print(f"Sklearn OHE matrix shape: {sklearn_ohe.shape}")
sklearn_ohe

Sklearn OHE matrix shape: (5, 53)


,already,amore,apply,around,available,buffet,bugis,c,cine,comp,crazy,cup,dont,dun,e,early,entry,fa,final,free,go,got,great,hor,joking,jurong,la,lar,life,may,n,nah,ok,oni,over,point,questionstd,ratetcs,receive,say,st,text,think,though,tkts,txt,u,usf,wat,wif,win,wkly,world
0,0,1,0,0,1,1,1,0,1,0,1,0,0,0,1,0,0,0,0,0,1,1,1,0,0,1,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0
2,0,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,1,1,1,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,1,1,1,0,1,1,0,0,1,1,0,0,0,0,1,1,0
3,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0
4,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,1,0,0,0,0,0


In [9]:
sklearn_ohe.to_csv("outputs/sklearn_one_hot.csv", index=False)
print("Saved: outputs/sklearn_one_hot.csv")

Saved: outputs/sklearn_one_hot.csv


---
# PART 2 — Bag of Words (BoW)

## Task 3: Bag of Words Representation

BoW counts how many times each vocabulary word appears in a document — unlike One-Hot which only stores presence/absence.

In [10]:
# Fit CountVectorizer on the full cleaned dataset
bow_vectorizer = CountVectorizer()
bow_matrix = bow_vectorizer.fit_transform(df["clean_text"])

print(f"BoW matrix shape: {bow_matrix.shape}")
print(f"  → {bow_matrix.shape[0]} messages × {bow_matrix.shape[1]} unique words")
print(f"\nSample vocabulary (first 20 words): {bow_vectorizer.get_feature_names_out()[:20]}")

BoW matrix shape: (5572, 7855)
  → 5572 messages × 7855 unique words

Sample vocabulary (first 20 words): ['aa' 'aah' 'aaniye' 'aaooooright' 'aathilove' 'aathiwhere' 'ab' 'abbey'
 'abdomen' 'abeg' 'abel' 'aberdeen' 'abi' 'ability' 'abiola' 'abj' 'able'
 'abnormally' 'aboutas' 'abroad']


In [11]:
# Convert sparse matrix to a DataFrame for inspection
bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=bow_vectorizer.get_feature_names_out()
)

print("Sample BoW vectors (first 5 rows, first 15 columns):")
bow_df.iloc[:5, :15]

Sample BoW vectors (first 5 rows, first 15 columns):


,aa,aah,aaniye,aaooooright,aathilove,aathiwhere,ab,abbey,abdomen,abeg,abel,aberdeen,abi,ability,abiola
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [12]:
bow_df.to_csv("outputs/bow_vectors.csv", index=False)
print("Saved: outputs/bow_vectors.csv")

Saved: outputs/bow_vectors.csv


## Task 4: Understanding Word Frequency

We sum each column (word) across all messages to find total occurrence counts.

In [13]:
# Sum each word column across all messages
word_counts = np.asarray(bow_matrix.sum(axis=0)).flatten()

# Map back to word names
word_freq_series = pd.Series(
    word_counts,
    index=bow_vectorizer.get_feature_names_out()
).sort_values(ascending=False)

print("Top 10 most frequent words in the dataset:")
print(word_freq_series.head(10))

print("\nTop 10 LEAST frequent words (appear only once):")
print(word_freq_series.tail(10))

Top 10 most frequent words in the dataset:
call    603
im      474
get     401
ur      384
go      308
dont    290
free    278
ok      277
ltgt    276
know    267
dtype: int64

Top 10 LEAST frequent words (appear only once):
kwish     1
kvb       1
kuch      1
ku        1
ktv       1
ksry      1
zahers    1
zac       1
ywhere    1
yupz      1
dtype: int64


In [14]:
# Explanation: why does BoW capture frequency but OHE doesn't?
print("=" * 55)
print("  BoW vs One-Hot: Frequency Comparison")
print("=" * 55)
print("""
One-Hot Encoding only records WHETHER a word is present (0 or 1).
If the word 'free' appears 3 times in a spam message, One-Hot
still stores just 1.

Bag of Words stores the actual COUNT. So 'free' appearing 3 times
is stored as 3. This frequency information is useful for classifiers
because spam messages tend to repeat certain keywords many times.
""")

  BoW vs One-Hot: Frequency Comparison

One-Hot Encoding only records WHETHER a word is present (0 or 1).
If the word 'free' appears 3 times in a spam message, One-Hot
still stores just 1.

Bag of Words stores the actual COUNT. So 'free' appearing 3 times
is stored as 3. This frequency information is useful for classifiers
because spam messages tend to repeat certain keywords many times.



---
# PART 3 — N-Grams

## Task 5: Unigrams, Bigrams & Trigrams

N-grams capture sequences of N words together — bigrams like "free entry" are far more meaningful than the two separate words.

In [15]:
# Unigrams — single words (same as BoW)
unigram_vec = CountVectorizer(ngram_range=(1, 1))
unigram_matrix = unigram_vec.fit_transform(df["clean_text"])

# Bigrams — pairs of consecutive words
bigram_vec = CountVectorizer(ngram_range=(2, 2))
bigram_matrix = bigram_vec.fit_transform(df["clean_text"])

# Trigrams — triples of consecutive words
trigram_vec = CountVectorizer(ngram_range=(3, 3))
trigram_matrix = trigram_vec.fit_transform(df["clean_text"])

print(f"Unigram vocabulary size : {unigram_matrix.shape[1]}")
print(f"Bigram  vocabulary size : {bigram_matrix.shape[1]}")
print(f"Trigram vocabulary size : {trigram_matrix.shape[1]}")

Unigram vocabulary size : 7855
Bigram  vocabulary size : 30085
Trigram vocabulary size : 30028


In [16]:
# Show sample features from each
print("Sample UNIGRAMS:", list(unigram_vec.get_feature_names_out()[:10]))
print("Sample BIGRAMS: ", list(bigram_vec.get_feature_names_out()[:10]))
print("Sample TRIGRAMS:", list(trigram_vec.get_feature_names_out()[:10]))

Sample UNIGRAMS: ['aa', 'aah', 'aaniye', 'aaooooright', 'aathilove', 'aathiwhere', 'ab', 'abbey', 'abdomen', 'abeg']
Sample BIGRAMS:  ['aa exhaust', 'aah bless', 'aah cuddle', 'aah speak', 'aaniye pudunga', 'aaooooright work', 'aathilove lot', 'aathiwhere dear', 'ab sara', 'abbey happy']
Sample TRIGRAMS: ['aa exhaust hanging', 'aah bless hows', 'aah cuddle would', 'aah speak tomo', 'aaniye pudunga venaam', 'ab sara jorgeshock', 'abbey happy new', 'abdomen gynae infection', 'abeg make profit', 'aberdeen united kingdom']


In [17]:
# Observation: bigram vocabulary is MUCH larger than unigram
# because every word pair becomes a feature
print("""
Key Observation:
  Unigrams capture individual words but miss context.
  'free' and 'win' are useful separately but 'free win' is
  a much stronger spam signal as a bigram.
  Trigrams add even more context but create a huge vocabulary.
""")


Key Observation:
  Unigrams capture individual words but miss context.
  'free' and 'win' are useful separately but 'free win' is
  a much stronger spam signal as a bigram.
  Trigrams add even more context but create a huge vocabulary.



## Task 6: Combined N-Grams (Unigrams + Bigrams)

Using `ngram_range=(1, 2)` we combine single words AND word pairs in one vector.

In [18]:
# Combine unigrams and bigrams into a single feature set
combined_vec = CountVectorizer(ngram_range=(1, 2))
combined_matrix = combined_vec.fit_transform(df["clean_text"])

print(f"Combined (1,2)-gram vocabulary size: {combined_matrix.shape[1]}")
print(f"  vs Unigram alone: {unigram_matrix.shape[1]}")
print(f"  → Adding bigrams adds {combined_matrix.shape[1] - unigram_matrix.shape[1]} extra features")
print()
print("Sample combined features (first 15):")
print(list(combined_vec.get_feature_names_out()[:15]))

Combined (1,2)-gram vocabulary size: 37940
  vs Unigram alone: 7855
  → Adding bigrams adds 30085 extra features

Sample combined features (first 15):
['aa', 'aa exhaust', 'aah', 'aah bless', 'aah cuddle', 'aah speak', 'aaniye', 'aaniye pudunga', 'aaooooright', 'aaooooright work', 'aathilove', 'aathilove lot', 'aathiwhere', 'aathiwhere dear', 'ab']


---
# PART 4 — TF-IDF Vectorization

## Task 7: TF-IDF Implementation

TF-IDF (Term Frequency–Inverse Document Frequency) down-weights common words  
that appear in many documents and highlights words that are unique to specific documents.

In [19]:
# Fit TF-IDF on the full dataset
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(df["clean_text"])

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"  → {tfidf_matrix.shape[0]} messages × {tfidf_matrix.shape[1]} unique words")
print()
print("Sample vocabulary (first 20):")
print(list(tfidf_vectorizer.get_feature_names_out()[:20]))

TF-IDF matrix shape: (5572, 7855)
  → 5572 messages × 7855 unique words

Sample vocabulary (first 20):
['aa', 'aah', 'aaniye', 'aaooooright', 'aathilove', 'aathiwhere', 'ab', 'abbey', 'abdomen', 'abeg', 'abel', 'aberdeen', 'abi', 'ability', 'abiola', 'abj', 'able', 'abnormally', 'aboutas', 'abroad']


In [20]:
# Display the TF-IDF matrix as a DataFrame
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out()
)

print("TF-IDF matrix (first 5 rows, first 12 columns):")
tfidf_df.iloc[:5, :12]

TF-IDF matrix (first 5 rows, first 12 columns):


,aa,aah,aaniye,aaooooright,aathilove,aathiwhere,ab,abbey,abdomen,abeg,abel,aberdeen
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [21]:
tfidf_df.to_csv("outputs/tfidf_vectors.csv", index=False)
print("Saved: outputs/tfidf_vectors.csv")

Saved: outputs/tfidf_vectors.csv


## Task 8: BoW vs TF-IDF Comparison

We compare the weights of common words between BoW and TF-IDF to understand the difference.

In [22]:
# Pick some common words to compare their BoW count vs TF-IDF score
comparison_words = ["free", "call", "text", "mobile", "get", "win", "claim"]

# Filter only words present in both vocabularies
bow_features    = list(bow_vectorizer.get_feature_names_out())
tfidf_features  = list(tfidf_vectorizer.get_feature_names_out())

comparison_rows = []

for word in comparison_words:
    if word in bow_features and word in tfidf_features:
        bow_total   = bow_matrix[:, bow_features.index(word)].sum()
        tfidf_mean  = tfidf_matrix[:, tfidf_features.index(word)].mean()
        comparison_rows.append({
            "word": word,
            "BoW total count": int(bow_total),
            "TF-IDF mean score": round(float(tfidf_mean), 6)
        })

comparison_df = pd.DataFrame(comparison_rows)
print(comparison_df.to_string(index=False))

  word  BoW total count  TF-IDF mean score
  free              278           0.009176
  call              603           0.021297
  text              215           0.008070
mobile              151           0.005319
   get              401           0.014240
   win               77           0.002949
 claim              115           0.004607


In [23]:
# Show words with highest TF-IDF scores (most distinctive words in the corpus)
tfidf_sums = np.asarray(tfidf_matrix.sum(axis=0)).flatten()
tfidf_word_scores = pd.Series(
    tfidf_sums,
    index=tfidf_vectorizer.get_feature_names_out()
).sort_values(ascending=False)

print("Words with HIGHEST total TF-IDF score (distinctive/important):")
print(tfidf_word_scores.head(10))

print("\nWords with LOWEST total TF-IDF score (very rare or very common):")
print(tfidf_word_scores.tail(10))

Words with HIGHEST total TF-IDF score (distinctive/important):
call    118.666810
ok       97.579413
im       95.880247
get      79.342885
ill      71.123699
come     65.862011
ur       65.818712
go       60.896271
dont     60.276228
ltgt     59.989346
dtype: float64

Words with LOWEST total TF-IDF score (very rare or very common):
friendshipmotherfatherteacherschildrens    0.097308
republic                                   0.097308
rememberi                                  0.097308
independence                               0.097308
shivratri                                  0.097308
approaching                                0.097308
dasara                                     0.097308
ugadi                                      0.097308
sankranti                                  0.097308
festival                                   0.097308
dtype: float64


In [24]:
print("""
Why TF-IDF down-weights common words:
--------------------------------------
A word like 'call' that appears in hundreds of messages gets
a high IDF penalty, pushing its TF-IDF score down.

A rare, specific word that appears in only a few spam messages
gets a LOW IDF penalty and thus a HIGHER TF-IDF score.

This is why TF-IDF generally outperforms raw BoW for text
classification tasks — it rewards uniqueness.
""")


Why TF-IDF down-weights common words:
--------------------------------------
A word like 'call' that appears in hundreds of messages gets
a high IDF penalty, pushing its TF-IDF score down.

A rare, specific word that appears in only a few spam messages
gets a LOW IDF penalty and thus a HIGHER TF-IDF score.

This is why TF-IDF generally outperforms raw BoW for text
classification tasks — it rewards uniqueness.



---
# PART 5 — Practical Insights

## Task 9: Vectorizer Parameter Exploration

We experiment with `max_features`, `min_df`, and `max_df` to see how vocabulary size changes.

In [25]:
experiments = [
    {"max_features": None, "min_df": 1,  "max_df": 1.0},   # default — all words
    {"max_features": 500,  "min_df": 1,  "max_df": 1.0},   # cap at 500 words
    {"max_features": None, "min_df": 5,  "max_df": 1.0},   # ignore very rare words
    {"max_features": None, "min_df": 1,  "max_df": 0.8},   # ignore very common words
    {"max_features": 1000, "min_df": 5,  "max_df": 0.9},   # balanced setting
]

print(f"{'max_features':>14}  {'min_df':>7}  {'max_df':>7}  {'vocab size':>12}")
print("-" * 50)

for exp in experiments:
    vec = TfidfVectorizer(
        max_features=exp["max_features"],
        min_df=exp["min_df"],
        max_df=exp["max_df"]
    )
    mat = vec.fit_transform(df["clean_text"])
    print(f"{str(exp['max_features']):>14}  {exp['min_df']:>7}  {exp['max_df']:>7}  {mat.shape[1]:>12}")

  max_features   min_df   max_df    vocab size
--------------------------------------------------
          None        1      1.0          7855
           500        1      1.0           500
          None        5      1.0          1521
          None        1      0.8          7855
          1000        5      0.9          1000


In [26]:
print("""
Parameter Observations:
------------------------
max_features : Caps the vocabulary to the N most frequent words.
               Useful to limit memory and speed up training.

min_df       : Ignores words that appear in fewer than N documents.
               Helps remove typos and very rare words that add noise.

max_df       : Ignores words that appear in MORE than X% of documents.
               Removes overly common words that don't help distinguish
               between classes (similar to stopword removal).
""")


Parameter Observations:
------------------------
max_features : Caps the vocabulary to the N most frequent words.
               Useful to limit memory and speed up training.

min_df       : Ignores words that appear in fewer than N documents.
               Helps remove typos and very rare words that add noise.

max_df       : Ignores words that appear in MORE than X% of documents.
               Removes overly common words that don't help distinguish
               between classes (similar to stopword removal).



## Task 10: Conceptual Questions

Brief answers to the four conceptual questions from the assignment.

In [27]:
print("""
Q1. Difference between One-Hot Encoding and Bag of Words:
----------------------------------------------------------
One-Hot Encoding records PRESENCE only (0 or 1) for each word.
It treats every word independently and ignores word frequency.

Bag of Words records COUNTS — how many times each word appears.
This frequency information is valuable for classification because
spam messages tend to repeat certain words (e.g. 'free', 'win') many times.


Q2. Why do N-grams increase dimensionality?
-------------------------------------------
Unigrams create one feature per word. Bigrams create one feature per
PAIR of consecutive words. The number of possible pairs is much larger
than the number of individual words. For V unique words, unigrams give
V features but bigrams can give up to V×V features in theory.
This explosion in feature count is what causes the dimensionality increase.


Q3. When to prefer TF-IDF over BoW:
-------------------------------------
Prefer TF-IDF when:
  - The dataset has many common words that appear frequently but carry
    little meaning (e.g. 'go', 'get', 'come').
  - You want to highlight words that are specific to certain documents.
  - Working on tasks like spam detection or topic classification where
    rare but specific keywords (e.g. 'prize', 'winner', 'claim') matter most.

Use raw BoW when:
  - Frequency itself is the signal (e.g. detecting repetitive spam).
  - The vocabulary is already well-curated after heavy preprocessing.


Q4. Limitations of count-based vectorization:
----------------------------------------------
1. No semantic understanding: 'good' and 'great' are treated as
   completely unrelated words.
2. Sparse matrices: most values are zero, wasting memory.
3. Word order is lost: 'not good' and 'good not' produce
   the same unigram vector.
4. Out-of-vocabulary words: any new word at inference time
   is simply ignored.
5. Fixed vocabulary: the model must be retrained to include new words.
""")


Q1. Difference between One-Hot Encoding and Bag of Words:
----------------------------------------------------------
One-Hot Encoding records PRESENCE only (0 or 1) for each word.
It treats every word independently and ignores word frequency.

Bag of Words records COUNTS — how many times each word appears.
This frequency information is valuable for classification because
spam messages tend to repeat certain words (e.g. 'free', 'win') many times.


Q2. Why do N-grams increase dimensionality?
-------------------------------------------
Unigrams create one feature per word. Bigrams create one feature per
PAIR of consecutive words. The number of possible pairs is much larger
than the number of individual words. For V unique words, unigrams give
V features but bigrams can give up to V×V features in theory.
This explosion in feature count is what causes the dimensionality increase.


Q3. When to prefer TF-IDF over BoW:
-------------------------------------
Prefer TF-IDF when:
  - The datase

---
# Final Insights

A summary of the most important observations from this assignment.

In [28]:
print("""
====================================================
  FINAL INSIGHTS — Assignment 18
====================================================

1. Dataset Balance:
   The SMS Spam dataset is heavily imbalanced — around 87% ham and
   13% spam. Any vectorization-based model trained on this data
   will need class balancing to avoid always predicting ham.

2. Preprocessing matters significantly:
   Lemmatization reduced words like 'running' → 'run' and
   'entries' → 'entry', keeping related words in the same bucket
   and reducing the vocabulary size noticeably.

3. One-Hot loses frequency information:
   The word 'free' in a spam message that says 'FREE FREE FREE'
   is encoded identically (value=1) to a message that says it once.
   Bag of Words correctly captures this as a count of 3.

4. BoW vocabulary is large but sparse:
   The full BoW matrix on 5,572 messages had over 7,000 features,
   but the average message only triggered a tiny fraction of them.
   Sparsity means most storage is wasted zeros.

5. Bigrams dramatically expand the feature space:
   Unigrams produced ~7,000 features while bigrams produced over
   35,000. This shows why N-gram models require careful feature
   limiting (max_features) for practical use.

6. Bigrams capture context unigrams miss:
   The bigram 'free entry' is a much stronger spam signal than
   'free' and 'entry' separately. N-grams help encode local
   word context without full sequence modeling.

7. TF-IDF down-weights generic words:
   Words like 'call' and 'get' that appear across almost every
   message got very low TF-IDF scores, while rare spam-specific
   words like 'prize' and 'claim' received higher scores.

8. min_df parameter is very useful in practice:
   Setting min_df=5 dropped hundreds of rare/misspelled words
   that would only add noise, reducing vocabulary by ~30%
   without losing meaningful features.

9. All methods lose word order:
   Both OHE and BoW treat 'I am not happy' identically to
   'I am happy not'. Trigrams partially mitigate this but
   not fully. True sequence awareness needs RNNs or Transformers.

10. TF-IDF is the best all-round starting point:
    For spam detection tasks, TF-IDF consistently outperforms raw
    BoW because it rewards distinctive spam keywords and penalizes
    generic words that appear in both ham and spam equally.
====================================================
""")


  FINAL INSIGHTS — Assignment 18

1. Dataset Balance:
   The SMS Spam dataset is heavily imbalanced — around 87% ham and
   13% spam. Any vectorization-based model trained on this data
   will need class balancing to avoid always predicting ham.

2. Preprocessing matters significantly:
   Lemmatization reduced words like 'running' → 'run' and
   'entries' → 'entry', keeping related words in the same bucket
   and reducing the vocabulary size noticeably.

3. One-Hot loses frequency information:
   The word 'free' in a spam message that says 'FREE FREE FREE'
   is encoded identically (value=1) to a message that says it once.
   Bag of Words correctly captures this as a count of 3.

4. BoW vocabulary is large but sparse:
   The full BoW matrix on 5,572 messages had over 7,000 features,
   but the average message only triggered a tiny fraction of them.
   Sparsity means most storage is wasted zeros.

5. Bigrams dramatically expand the feature space:
   Unigrams produced ~7,000 features wh